In [45]:
import os
import requests
import gradio as gr
from dotenv import load_dotenv

load_dotenv()
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
MODELS_URL = "https://openrouter.ai/api/v1/models"
chatbot_ui = gr.Chatbot()

download_btn = gr.Button("Download Conversation")
download_file = gr.File(label="Download")

In [42]:
def list_free_models():
    try:
        r = requests.get(MODELS_URL, timeout=15)
        r.raise_for_status()
        all_models = r.json()["data"]
        free_ids = [m["id"] for m in all_models if m["id"].endswith(":free")]
        free_ids.sort()
    except Exception:
        free_ids = []
    # openrouter/free is a built-in router that always works, so it's always first
    return ["openrouter/free"] + free_ids


In [44]:
def download_conversation(history):
    if not history:
        return None

    conversation = ""

    for message in history:
        if isinstance(message, dict):
            role = message.get("role", "unknown")
            content = message.get("content", "")
        else:
            # For older Gradio chatbot history format
            user_message, bot_message = message
            conversation += f"User: {user_message}\n"
            conversation += f"Assistant: {bot_message}\n\n"
            continue

        if role == "user":
            conversation += f"User: {content}\n"
        elif role == "assistant":
            conversation += f"Assistant: {content}\n"

    with open("conversation.txt", "w", encoding="utf-8") as file:
        file.write(conversation)

    return "conversation.txt"

In [43]:
def ask_openrouter(messages, model):
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "HTTP-Referer": "https://jekacode.africa",
        "X-Title": "AI Chatbot",
    }
    payload = {"model": model, "messages": messages}
    r = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=60)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"].strip()


In [36]:
def chat_fn(message, history, models):

    if not models:
        return "Please select at least one model."

    messages = []

    for turn in history:
        messages.append({
            "role": turn["role"],
            "content": turn["content"]
        })

    messages.append({
        "role": "user",
        "content": message
    })

    responses = {}

    for model in models:
        try:
            response = ask_openrouter(messages, model)
            responses[model] = response

        except requests.exceptions.HTTPError as e:
            responses[model] = f"Error: {e}"

        except Exception as e:
            responses[model] = f"Something went wrong: {e}"

    return "\n\n".join(
        f"### {model}\n{response}"
        for model, response in responses.items()
    )

In [37]:
test_responses = {
    "Model A": "This is Model A's response.",
    "Model B": "This is Model B's response."
}

print(test_responses)

{'Model A': "This is Model A's response.", 'Model B': "This is Model B's response."}


In [38]:
def cross_analyze(responses):
    analysis = "\n\n".join(
        f"MODEL: {model}\nRESPONSE:\n{response}"
        for model, response in responses.items()
    )

    prompt = f"""
Compare the following AI model responses.

{analysis}

Give a concise cross-analysis covering:
1. Accuracy
2. Clarity
3. Completeness
4. Strengths
5. Weaknesses

Finally, state which response is strongest overall and briefly explain why.
"""

    return ask_openrouter(
        [{"role": "user", "content": prompt}],
        "openrouter/free"
    )

In [48]:
free_models = list_free_models()
default_model = free_models[0] if free_models else "openrouter/free"

with gr.Blocks() as demo:

    model_picker = gr.Dropdown(
        choices=free_models,
        value=[default_model],
        multiselect=True,
        label="Select Models (free tier)",
    )

    chat = gr.ChatInterface(
        fn=chat_fn,
        additional_inputs=[model_picker],
        title="AI Chatbot — free models via OpenRouter",
        description="Pick free models, then chat.",
    )

    cross_analysis_btn = gr.Button("🔍 Run Cross-Analysis")

    analysis_output = gr.Markdown()

    cross_analysis_btn.click(
        fn=lambda: cross_analyze(latest_responses),
        inputs=[],
        outputs=analysis_output
    )

    # Download Conversation
    download_btn = gr.Button("📥 Download Conversation")
    download_file = gr.File(label="Download")

    download_btn.click(
        fn=download_conversation,
        inputs=chat,
        outputs=download_file
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.
